<!-- AI-GENERATED
     Model   : Claude Sonnet 4.6
     Date    : 2026-05-08
     Prompt  : can you do this notebook in the expirements uivenus folder with the name runner_on_kaggle
-->

# UI-Venus ScreenSpot-Pro Runner (Kaggle)

This notebook runs full grounding and evaluation for UI-Venus on ScreenSpot-Pro.
Outputs are saved inside this experiment folder.

In [15]:
%pip install -q pydantic pillow pyyaml loguru huggingface-hub transformers qwen-vl-utils accelerate

Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path('/kaggle/working/Click2Act')
REPO_BRANCH = 'uivenus'
REPO_URL_PUBLIC = 'https://github.com/YaraHisham61/Click2Act.git'

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_ACCESS_TOKEN')
except Exception:
    token = None

if token:
    repo_url = f'https://{token}@github.com/YaraHisham61/Click2Act.git'
else:
    repo_url = REPO_URL_PUBLIC

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, repo_url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Working directory:', REPO_DIR)
print('Branch:', REPO_BRANCH)

Your branch is up to date with 'origin/uivenus'.
HEAD is now at 22bf7a2 feat(eval): add robust uivenus full-run and eval pipeline
Working directory: /kaggle/working/Click2Act
Branch: uivenus


From https://github.com/YaraHisham61/Click2Act
 * branch            uivenus    -> FETCH_HEAD
Already on 'uivenus'


In [17]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('cuda name:', torch.cuda.get_device_name(0))
    print('compute capability:', torch.cuda.get_device_capability(0))

!nvidia-smi

torch: 2.10.0+cu128
cuda available: True
cuda device count: 2
cuda name: Tesla T4
compute capability: (7, 5)
Thu May  7 22:19:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             36W /   70W |    7405MiB /  15360MiB |      0%      Default |
|                              

In [18]:
from pathlib import Path

EXP_DIR = Path('experiments/2026-04-27_exp_uivenus_screenspotpro')
GROUNDER_CFG = EXP_DIR / 'grounder.yaml'
EVAL_CFG = EXP_DIR / 'evaluate_grounder.yaml'
GROUNDER_CSV = EXP_DIR / 'grounder.csv'
EVAL_SUMMARY = EXP_DIR / 'eval_summary.json'
VALID_ONLY_CSV = EXP_DIR / 'grounder_valid_only.csv'

print('Grounder config:', GROUNDER_CFG)
print('Eval config:', EVAL_CFG)
print('Grounder CSV:', GROUNDER_CSV)
print('Eval summary:', EVAL_SUMMARY)

Grounder config: experiments/2026-04-27_exp_uivenus_screenspotpro/grounder.yaml
Eval config: experiments/2026-04-27_exp_uivenus_screenspotpro/evaluate_grounder.yaml
Grounder CSV: experiments/2026-04-27_exp_uivenus_screenspotpro/grounder.csv
Eval summary: experiments/2026-04-27_exp_uivenus_screenspotpro/eval_summary.json


In [ ]:
# Full run (resume-safe).
# OOM mitigation: use expandable segments allocator for long runs.
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!python -m src.pipeline.grounder "experiments/2026-04-27_exp_uivenus_screenspotpro/grounder.yaml"

[uivenus-debug] Loading model and processor...
Loading weights: 100%|█| 729/729 [00:17<00:00, 42.30it/s, Materializing param=mo
Some parameters are on the meta device because they were offloaded to the cpu.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[uivenus-debug] Model and processor loaded.
batches: 0it [00:00, ?it/s]
2026-05-07 22:19:39.380 | INFO     | __main__:main:155 - Grounder run complete | total=0 ok=0 failed=0 timeout=0 output_csv=experiments/2026-04-27_exp_uivenus_screenspotpro/grounder.csv


In [20]:
# Evaluate only after grounding output is available.
from pathlib import Path

grounder_csv = Path('experiments/2026-04-27_exp_uivenus_screenspotpro/grounder.csv')
if not grounder_csv.exists():
    raise FileNotFoundError(
        f"{grounder_csv} not found. Run the grounding cell above first (or resume it)."
    )

!python experiments/2026-04-27_exp_uivenus_screenspotpro/run_evaluation.py

Traceback (most recent call last):
  File "/kaggle/working/Click2Act/src/pipeline/evaluate_grounder.py", line 104, in <module>
    main(config)
  File "/kaggle/working/Click2Act/src/pipeline/evaluate_grounder.py", line 45, in main
    valid_df["point_in_bbox"] = valid_df.apply(_point_in_bbox, axis=1)
    ~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py", line 4312, in __setitem__
    self._set_item_frame_value(key, value)
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py", line 4470, in _set_item_frame_value
    raise ValueError(
ValueError: Cannot set a DataFrame with multiple columns to the single column point_in_bbox


In [21]:
import json
import pandas as pd
from pathlib import Path

summary_path = Path('experiments/2026-04-27_exp_uivenus_screenspotpro/eval_summary.json')
valid_csv_path = Path('experiments/2026-04-27_exp_uivenus_screenspotpro/grounder_valid_only.csv')
grounder_csv_path = Path('experiments/2026-04-27_exp_uivenus_screenspotpro/grounder.csv')

print('grounder.csv exists:', grounder_csv_path.exists())
print('grounder_valid_only.csv exists:', valid_csv_path.exists())
print('eval_summary.json exists:', summary_path.exists())

if grounder_csv_path.exists():
    df = pd.read_csv(grounder_csv_path)
    print('rows in grounder.csv:', len(df))
    if 'status' in df.columns:
        print('status counts:')
        print(df['status'].value_counts(dropna=False))
    if 'failed_reason' in df.columns:
        print('\nfailed_reason counts:')
        print(df['failed_reason'].fillna('<none>').value_counts().head(20))

if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))

grounder.csv exists: True
grounder_valid_only.csv exists: False
eval_summary.json exists: False
rows in grounder.csv: 1581
status counts:
status
failed    1581
Name: count, dtype: int64

failed_reason counts:
failed_reason
OutOfMemoryError('CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1011.81 MiB is free. Process 104 has 7.23 GiB memory in use. Including non-PyTorch memory, this process has 6.34 GiB memory in use. Of the allocated memory 6.20 GiB is allocated by PyTorch, and 21.50 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)')    304
OutOfMemoryError('CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1011.81 MiB is free. Process 104 has 7.23 GiB memory in 